<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/ace_step_1-5-customi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, shutil
if os.path.exists("Ace-Step-v1.5"): shutil.rmtree("Ace-Step-v1.5")
!git clone https://huggingface.co/spaces/ACE-Step/Ace-Step-v1.5
os.chdir("/content/Ace-Step-v1.5")
!sed -i 's/torch>=2.9.1/torch/g' requirements.txt
!sed -i 's/share=False/share=True/g' app.py
print("✅ Step 1 xong.")

Cloning into 'Ace-Step-v1.5'...
remote: Enumerating objects: 1298, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 1298 (delta 0), reused 0 (delta 0), pack-reused 1295 (from 1)
Receiving objects: 100% (1298/1298), 1.57 MiB | 5.88 MiB/s, done.
Resolving deltas: 100% (789/789), done.
✅ Step 1 xong.


In [2]:
import os
os.chdir("/content/Ace-Step-v1.5")
!pip install --no-cache-dir -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
!pip install --no-cache-dir ffmpeg-python
!apt-get install -y ffmpeg
print("✅ Step 2 xong. Bấm RESTART SESSION nếu Colab yêu cầu, rồi chạy Ô 3.")

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu128
Ignoring torch: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchaudio: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchvision: markers 'sys_platform == "win32"' don't match your environment
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "win32" and python_version == "3.11" and platform_machine == "AMD64"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "linux" and python_version == "3.11"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 209.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 169.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versio

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
✅ Step 2 xong. Bấm RESTART SESSION nếu Colab yêu cầu, rồi chạy Ô 3.


In [3]:
# import os
# os.chdir("/content/Ace-Step-v1.5")
# print("✅ Đã quay lại thư mục làm việc.")

In [ ]:
import os
import shutil

# --- BƯỚC 1: XÓA DỮ LIỆU LỖI ---
print("🧹 Đang dọn dẹp các thư mục model bị hỏng...")
paths_to_clean = ["/content/Ace-Step-v1.5/data", "/root/.cache/huggingface"]
for path in paths_to_clean:
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"✅ Đã xóa: {path}")

# --- BƯỚC 2: CÀI ĐẶT NANO-VLLM THỦ CÔNG ---
print("🚀 Đang cài đặt nano-vllm để tăng tốc độ...")
os.chdir("/content/Ace-Step-v1.5/acestep/third_parts/nano-vllm")
!pip install .
os.chdir("/content/Ace-Step-v1.5")

# --- BƯỚC 3: ÉP TẢI LẠI MODEL CHUẨN ---
print("⏳ Bắt đầu tải Model 10GB (Đừng tắt tab này!)...")
download_script = """
import os
import sys
from acestep.handler import AceStepHandler
current_dir = os.getcwd()
sys.path.insert(0, os.path.join(current_dir, "acestep", "third_parts", "nano-vllm"))

# Khởi tạo và ép tải model về thư mục data
handler = AceStepHandler(persistent_storage_path=os.path.join(current_dir, "data"))
config_path = "acestep-v15-turbo"
try:
    print("📥 Downloading DiT & LM models...")
    handler.initialize_service(current_dir, config_path, device='cpu', offload_to_cpu=True)
except Exception as e:
    print(f"ℹ️ Status: {e}")
"""

with open("force_dl.py", "w") as f:
    f.write(download_script)

!python force_dl.py

# --- BƯỚC 4: THIẾT LẬP RAM & CHẠY ---
os.environ["SERVICE_MODE_DIT_MODEL_2"] = ""
os.environ["VLLM_GPU_MEMORY_UTILIZATION"] = "0.5" # Chống tràn RAM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("="*60)
print("🎹 MODEL ĐÃ SẴN SÀNG. ĐANG KHỞI ĐỘNG SERVER...")
print("="*60)
!python app.py

🧹 Đang dọn dẹp các thư mục model bị hỏng...
✅ Đã xóa: /content/Ace-Step-v1.5/data
✅ Đã xóa: /root/.cache/huggingface
🚀 Đang cài đặt nano-vllm để tăng tốc độ...
Processing /content/Ace-Step-v1.5/acestep/third_parts/nano-vllm
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 73.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for nano-vllm: filename=nano_vllm-0.2.0-py3-none-any.whl size=29188 sha256=3357fa7f83d046478de045b60d8112e12cd803221eef8d6c576d734a06dde6ba
  Stored in directory: /root/.cache/pip/wheels/0c/95/fd/55ed745fe219a1900e427f7db76e5eef5c2e3c3e0cfdaa8a7e
